In [ ]:
# original MNIST
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# TODO demo dataset, please change!
x = pd.read_csv("./mnist-imgs-mini.csv.bz2", index_col=0)
x = x/255.0
x = x.to_numpy()
print(x.shape, x.dtype)

y = pd.read_csv("./mnist-lbls-mini.csv.bz2", index_col=0)
y = y.to_numpy()
print(y.shape)

label_names = range(10)
labels = list(label_names)
print(label_names)

n_channels = 1

In [ ]:

# distance function for two flattened images (euclidean distance)
def distance(img1, img2):
    return np.linalg.norm(img1 - img2)

# compute distance matrix for all pairs of images
def compute_distance_matrix(X):
    n_samples = X.shape[0]
    distance_matrix = np.zeros((n_samples, n_samples))
    for i in range(n_samples):
        for j in range(i + 1, n_samples):
            dist = distance(X[i], X[j])
            distance_matrix[i, j] = dist
            distance_matrix[j, i] = dist
    return distance_matrix

# Ensure labels are a flat vector
y_flat = y.ravel().astype(int)

n_features = x.shape[1]
img_side = int(np.sqrt(n_features))
if img_side * img_side != n_features:
    raise ValueError(f"Expected square image features, got {n_features} features.")

centroids = []
counts = []


for digit in range(10):
    x_digit = x[y_flat == digit]
    counts.append(x_digit.shape[0])
    
    # use 10% of each digit
    cluster_size = x_digit.shape[0] // 10

    distances = compute_distance_matrix(x_digit)
    np.fill_diagonal(distances, np.inf)  # ignore self-distances

    # implement hierarchical clustering with complete linkage without using sklearn
    # start with smallest distance in distance matrix and build a single cluster by merging the closest images, only 
    # (no other clusters as there is only a single cluster)
    # loop for cluster_size iterations and collect the merged rows
    collected_indices = set()
    # find the indices of the minimum distance in the distance matrix
    i, j = np.unravel_index(np.argmin(distances), distances.shape)
    collected_indices.add(i)
    collected_indices.add(j)
    # two have been pushed, already
    for _ in range(cluster_size - 2):
        # select only the rows of the distance matrix that correspond to the collected indices
        collected_distances = distances[list(collected_indices), :]
        # reduce to maximum distance for complete linkage
        max_distances = collected_distances.max(axis=0)
        # set the distances of the collected indices to infinity to ignore them
        max_distances[list(collected_indices)] = np.inf
        # find the index of the minimum distance in the reduced distance matrix
        next_index = np.argmin(max_distances)
        collected_indices.add(next_index)
    
    # after the loop has finished, compute the centroid of the merged cluster and store it in centroids
    if not collected_indices:
        raise ValueError(f"No samples were collected for digit {digit}.")
    
    print(f'{len(collected_indices)} collected for digit {digit}')

    cluster_points = x_digit[list(collected_indices)]
    centroid = cluster_points.mean(axis=0).reshape(img_side, img_side)
    centroids.append(centroid)

print("Samples per digit:", counts)



In [ ]:
# store centroids in pd frame and store as csv
centroids_df = pd.DataFrame([centroid.flatten() for centroid in centroids], index=labels)
centroids_df.to_csv("centroids.csv.bz2", compression="bz2")

In [ ]:
# load centroids from csv
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

centroids_df = pd.read_csv("centroids.csv.bz2", index_col=0)
n_features = centroids_df.shape[1]
img_side = int(np.sqrt(n_features))

centroids = centroids_df.to_numpy().reshape(-1, img_side, img_side)

In [ ]:
# plot the centroids
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit, ax in enumerate(axes.flat):
    ax.imshow(1.0 - centroids[digit], cmap="gray")
    #ax.set_title(f"Digit {digit}")
    ax.axis("off")

#plt.suptitle("MNIST per-digit centroids")
plt.tight_layout()
plt.show()